In [0]:
# ==================================================
# GOLD — Causas de Acidentes
# Pipeline PRF Acidentes de Trânsito
# ==================================================

from pyspark.sql.functions import col, sum, count, round, desc

# Lê da Silver
df = spark.table("workspace.default.silver_acidentes")

# Total geral para calcular percentual
total = df.count()

# Agrega por causa
df_gold_causas = (df
    .groupBy("causa_acidente")
    .agg(
        count("id").alias("total_acidentes"),
        sum("mortos").alias("total_mortos"),
        sum("feridos").alias("total_feridos")
    )
    .withColumn("percentual_acidentes",
        round((col("total_acidentes") / total) * 100, 2))
    .withColumn("letalidade",
        round(col("total_mortos") / col("total_acidentes"), 4))
    .orderBy(desc("total_acidentes"))
)

# Grava Gold
(df_gold_causas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_causas")
)

print("Gold causas gravada!")
df_gold_causas.show(10, truncate=False)